# Part3 Stage1 Difix Dataset Prep (Re10k-1 / RegGS)

这个 notebook 是主脚本  的薄调用层。
建议在 **reggs 环境** 下打开运行。


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

def run_cmd(cmd):
    print(' ' .join(cmd))
    proc = subprocess.run(cmd, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout)
    if proc.returncode != 0:
        if proc.stderr:
            print(proc.stderr)
        raise RuntimeError(f'Command failed with code {proc.returncode}')
    return proc

def read_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


In [ ]:
PYTHON = sys.executable  # 建议 notebook kernel 就是 reggs env
SCRIPT = Path('/home/bzhang512/CV_Project/part3_BRPO/scripts/prepare_stage1_difix_dataset.py')

SCENE_NAME = 'Re10k-1'
BACKEND = 'reggs'
RUN_KEY = 're10k1__reggs__midpoint__v1'
DATASET_ROOT = Path('/home/bzhang512/CV_Project/dataset')
RUN_ROOT = DATASET_ROOT / SCENE_NAME / 'part3_stage1' / RUN_KEY

TRAIN_DIR = Path('/home/bzhang512/CV_Project/output/part2/re10k_1/reggs_re10k1_re10k-ckpt_sr50_nv9_sm2_comparison_check/train')
RENDER_DIR = Path('/home/bzhang512/CV_Project/output/part2/re10k_1/reggs_re10k1_re10k-ckpt_sr50_nv9_sm2_comparison_check/test_all_non_train_subset_v1')

PLACEMENT = 'midpoint'   # midpoint | tertile | both | manual
MANUAL_IDS = None        # 例如 '17,52,88'
PROMPT = 'remove degradation'
DIFIX_MODEL_NAME = 'nvidia/difix_ref'
HEIGHT = 512
WIDTH = 512
TIMESTEP = 199

print('RUN_ROOT =', RUN_ROOT)
print('PYTHON   =', PYTHON)
print('SCRIPT   =', SCRIPT)


In [ ]:
# SELECT
cmd = [
    PYTHON, str(SCRIPT),
    '--stage', 'select',
    '--backend', BACKEND,
    '--scene-name', SCENE_NAME,
    '--train-dir', str(TRAIN_DIR),
    '--render-dir', str(RENDER_DIR),
    '--dataset-root', str(DATASET_ROOT),
    '--run-key', RUN_KEY,
    '--placement', PLACEMENT,
]
if MANUAL_IDS:
    cmd += ['--manual-ids', MANUAL_IDS]
run_cmd(cmd)
summary = read_json(RUN_ROOT / 'manifests' / 'selection_summary.json')
summary


In [ ]:
# DIFIX
cmd = [
    PYTHON, str(SCRIPT),
    '--stage', 'difix',
    '--backend', BACKEND,
    '--scene-name', SCENE_NAME,
    '--dataset-root', str(DATASET_ROOT),
    '--run-key', RUN_KEY,
    '--difix-model-name', DIFIX_MODEL_NAME,
    '--prompt', PROMPT,
    '--height', str(HEIGHT),
    '--width', str(WIDTH),
    '--timestep', str(TIMESTEP),
]
run_cmd(cmd)
difix_manifest = read_json(RUN_ROOT / 'manifests' / 'difix_run_manifest.json')
difix_manifest


In [ ]:
# PACK
cmd = [
    PYTHON, str(SCRIPT),
    '--stage', 'pack',
    '--backend', BACKEND,
    '--scene-name', SCENE_NAME,
    '--dataset-root', str(DATASET_ROOT),
    '--run-key', RUN_KEY,
]
run_cmd(cmd)
pack_manifest = read_json(RUN_ROOT / 'manifests' / 'pack_manifest.json')
pack_manifest


In [ ]:
# QUICK CHECK
left_dir = RUN_ROOT / 'difix' / 'left_fixed'
right_dir = RUN_ROOT / 'difix' / 'right_fixed'
aug_left = RUN_ROOT / 'augmented_train_left' / 'rgb'
aug_right = RUN_ROOT / 'augmented_train_right' / 'rgb'

print('left_fixed_count  =', len(list(left_dir.glob('*.png'))))
print('right_fixed_count =', len(list(right_dir.glob('*.png'))))
print('aug_left_count    =', len(list(aug_left.glob('*.png'))))
print('aug_right_count   =', len(list(aug_right.glob('*.png'))))
print('sample left  =', sorted(p.name for p in left_dir.glob('*.png'))[:5])
print('sample right =', sorted(p.name for p in right_dir.glob('*.png'))[:5])
